In [12]:
# %% [markdown]
# # Warm-Start Analysis: live reference lookup
# For each W* run, find the cold reference of the same base method
# (WCRAB↔CRAB, WGRAPE↔GRAPE) matching `(num_tslots, detuning, drive_error)`
# in the same directory. Report additional iterations and SI improvement.
#
# Claim: for WCRAB, ~2 iter/state give ~100x SI improvement.

# %%
from pathlib import Path
from collections import defaultdict
import numpy as np

from veripulse.pulse import PulseResult
#from utils import collect_files          # your helper
# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
DATA_DIR = Path.cwd().parent / "data/dummyless"    # or "data/dummyyes"
BASE_OF  = {"WCRAB": "CRAB", "WGRAPE": "GRAPE"}    # warm -> cold pairing


# %%
# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
# data folder
#data_dir_dummyyes = Path.cwd().parent/"data/dummyyes"
#data_dir_dummyless = Path.cwd().parent/"data/dummyless"


# file naming pattern
_PATTERN = re.compile(
    r"^(?P<method>[^_]+)_p(?P<num_tslots>\d+)_det(?P<detuning>-?[\d.]+)_err(?P<drive_error>-?[\d.]+)-(?P<identification>.+)\.json$"
)

# GRAPE_AVG:  GRAPE_AVG_pN_detX.XX_errX.XX-lamX.X-ID.json
_PATTERN_AVG = re.compile(
    r"^(?P<method>GRAPE_AVG)_p(?P<num_tslots>\d+)_det(?P<detuning>-?[\d.]+)_err(?P<drive_error>-?[\d.]+)-lam(?P<lam>[\d.]+)-(?P<identification>.+)\.json$"
)
_PATTERN_WARM = re.compile(
    r"^(?P<method>W(?:GRAPE|CRAB))_p(?P<num_tslots>\d+)_det(?P<detuning>-?[\d.]+)_err(?P<drive_error>-?[\d.]+)-lam(?P<lam>[\d.]+)-(?P<identification>.+)\.json$"
)

# parsers
def parse_filename(path: Path) -> Optional[dict]:
    m = _PATTERN_AVG.match(path.name) \
        or _PATTERN_WARM.match(path.name) \
        or _PATTERN.match(path.name)
    if not m:
        return None
    d = {
        "path":           path,
        "method":         m.group("method"),
        "num_tslots":     int(m.group("num_tslots")),
        "detuning":       float(m.group("detuning")),
        "drive_error":    float(m.group("drive_error")),
        "identification": m.group("identification"),
        "lam":            float(m.group("lam")) if "lam" in m.groupdict() and m.group("lam") is not None else None,
    }
    return d


def collect_files(
    directory: str | Path,
    method: Optional[str] = None,
    num_tslots: Optional[int] = None,
    detuning: Optional[float] = None,
    drive_error: Optional[float] = None,
    identification: Optional[str] = None,
    lam: Optional[float] = None,
    load: bool = False,
) -> list[dict]:
    results = []
    for path in sorted(Path(directory).glob("*.json")):
        meta = parse_filename(path)
        if meta is None:
            continue

        if method        is not None and meta["method"]      != method:                               continue
        if num_tslots    is not None and meta["num_tslots"]  != num_tslots:                           continue
        if detuning      is not None and f"{meta['detuning']:.2f}"    != f"{detuning:.2f}":           continue
        if drive_error   is not None and f"{meta['drive_error']:.2f}" != f"{drive_error:.2f}":        continue
        if identification is not None and meta["identification"] != str(identification):               continue
        if lam           is not None and (meta["lam"] is None or f"{meta['lam']:.3f}" != f"{lam:.3f}"): continue

        if load:
            with open(path) as f:
                meta["data"] = json.load(f)

        results.append(meta)

    return results

def iters_per_state(res) -> list[int]:
    """List-per-state (CRAB/GRAPE) or scalar (GRAPE_AVG)."""
    fcl = res.fid_compute_list
    if np.isscalar(fcl):
        return [int(fcl)]
    return [int(x) for x in fcl]


def find_cold_reference(meta_warm: dict) -> dict | None:
    """
    Match a W* run to its cold counterpart in the same directory
    on (num_tslots, detuning, drive_error). Picks the run with the
    lowest SI when several candidates exist.
    """
    base = BASE_OF.get(meta_warm["method"])
    if base is None:
        return None

    cands = collect_files(
        directory   = DATA_DIR,
        method      = base,
        num_tslots  = meta_warm["num_tslots"],
        detuning    = meta_warm["detuning"],
        drive_error = meta_warm["drive_error"],
    )
    if not cands:
        return None

    best = None
    for c in cands:
        try:
            res = PulseResult.load(c["path"])
        except Exception:
            continue
        si = float(res.si)
        if best is None or si < best[0]:
            best = (si, c["path"].name, res)
    return None if best is None else {"si": best[0],
                                      "file": best[1],
                                      "res": best[2]}



# %%
# ------------------------------------------------------------------
# Aggregated summary per method
# ------------------------------------------------------------------
def summarise(records: list[dict]) -> dict:
    it   = np.array([r["iter_per_state_mean"] for r in records])
    itmx = np.array([r["iter_per_state_max"]  for r in records])
    sa   = np.array([r["si_after"] for r in records])
    sb   = np.array([r["si_before"]   for r in records if r["si_before"]   is not None])
    imp  = np.array([r["improvement"] for r in records if r["improvement"] is not None])
    return {
        "n_runs":            len(records),
        "iter/state avg":    float(it.mean()),
        "iter/state median": float(np.median(it)),
        "iter/state max":    int(itmx.max()),
        "si before avg":     float(sb.mean())  if len(sb) else None,
        "si after best":     float(sa.min()),
        "si after avg":      float(sa.mean()),
        "improvement best":  float(imp.max())  if len(imp) else None,
        "improvement avg":   float(imp.mean()) if len(imp) else None,
    }


In [11]:
summaries = {m: summarise(recs) for m, recs in by_method.items()}

print("\n--- Aggregated summary ---")
cols = list(next(iter(summaries.values())).keys())
w = max(len(m) for m in summaries) + 2
print(f"{'method':<{w}}" + "  ".join(f"{c:>17}" for c in cols))
print("-" * (w + 19 * len(cols)))
for m, s in summaries.items():
    row = []
    for c in cols:
        v = s[c]
        row.append(f"{v:>17.3g}" if isinstance(v, float) else f"{v!s:>17}")
    print(f"{m:<{w}}" + "  ".join(row))


# %%
# ------------------------------------------------------------------
# Claim check
# ------------------------------------------------------------------
print("\n--- Claim check: ~2 iter/state -> ~100x SI improvement ---")
for m, s in summaries.items():
    it = s["iter/state avg"]
    imp = s["improvement best"]
    imp_avg = s["improvement avg"]
    if imp is None:
        print(f"  {m}: {it:.1f} iter/state, no reference matches found")
        continue
    verdict = "SUPPORTED" if it <= 3 and imp >= 50 else "not the 100x claim"
    print(f"  {m}: {it:.1f} iter/state on average  "
          f"-> best {imp:.0f}x, mean {imp_avg:.0f}x SI improvement  [{verdict}]")


# %%
# ------------------------------------------------------------------
# LaTeX row for the paper
# ------------------------------------------------------------------
print("\n% LaTeX (paste into a table)")
print(r"\begin{tabular}{lrrrr}")
print(r"Method & Avg. iter/state & $\bar\varepsilon$ before & $\bar\varepsilon$ after & Improvement \\")
print(r"\hline")
for m, s in summaries.items():
    it  = s["iter/state avg"]
    b   = s["si before avg"]
    a   = s["si after best"]
    imp = s["improvement best"]
    b_s   = f"${b:.2g}$"   if b   is not None else "--"
    imp_s = f"${imp:.0f}$" if imp is not None else "--"
    print(f"{m} & {it:.1f} & {b_s} & ${a:.2g}$ & {imp_s} \\\\")
print(r"\end{tabular}")


--- Aggregated summary ---
method             n_runs     iter/state avg  iter/state median     iter/state max      si before avg      si after best       si after avg   improvement best    improvement avg
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
WCRAB                 100                 21                 20                 65             0.0489            0.00058             0.0029               76.4               36.8
WGRAPE                100               2.77                  2                 44            0.00114           0.000977            0.00161                  1              0.907

--- Claim check: ~2 iter/state -> ~100x SI improvement ---
  WCRAB: 21.0 iter/state on average  -> best 76x, mean 37x SI improvement  [not the 100x claim]
  WGRAPE: 2.8 iter/state on average  -> best 1x, mean 1x SI improvement  [not the 100x claim]

% LaTeX

In [13]:
# %%
# ------------------------------------------------------------------
# Pair every W* with its reference
# ------------------------------------------------------------------
warm_files = collect_files(DATA_DIR)   # everything; we'll filter
warm_files = [m for m in warm_files if m["method"] in BASE_OF]
print(f"Found {len(warm_files)} W* runs in {DATA_DIR}")

by_method = defaultdict(list)
for meta in warm_files:
    try:
        res_w = PulseResult.load(meta["path"])
    except Exception as e:
        print(f"  [skip] {meta['path'].name}: {e}")
        continue

    ref = find_cold_reference(meta)
    it  = iters_per_state(res_w)

    by_method[meta["method"]].append({
        "file":               meta["path"].name,
        "num_tslots":         meta["num_tslots"],
        "detuning":           meta["detuning"],
        "drive_error":        meta["drive_error"],
        "lam":                meta.get("lam"),
        "iter_per_state_mean": float(np.mean(it)),
        "iter_per_state_max":  int(np.max(it)),
        "iter_total":          int(np.sum(it)),
        "si_after":            float(res_w.si),
        "si_before":           ref["si"] if ref else None,
        "ref_file":            ref["file"] if ref else None,
        "improvement":         (ref["si"] / float(res_w.si)) if ref else None,
    })

for m, recs in by_method.items():
    matched = sum(1 for r in recs if r["si_before"] is not None)
    print(f"  {m}: {len(recs)} runs, {matched} matched to a cold reference")


# %%
# ------------------------------------------------------------------
# Per-file table
# ------------------------------------------------------------------
print()
w = 55
hdr = (f"{'file':<{w}}  {'lam':>6}  {'iter/state':>10}  "
       f"{'si before':>10}  {'si after':>10}  {'improve':>10}")
print(hdr); print("-" * len(hdr))
for m, recs in by_method.items():
    for r in sorted(recs, key=lambda x: (x["num_tslots"], x["lam"] or 0, x["file"])):
        lam = f"{r['lam']:.3f}" if r["lam"] is not None else "  --"
        b   = f"{r['si_before']:.2e}"   if r["si_before"]   is not None else "   n/a"
        imp = f"{r['improvement']:.1f}x" if r["improvement"] is not None else "   n/a"
        print(f"{r['file']:<{w}}  {lam:>6}  "
              f"{r['iter_per_state_mean']:>10.1f}  "
              f"{b:>10}  {r['si_after']:>10.2e}  {imp:>10}")


Found 200 W* runs in /users/home/gustiani/VeriPulse/data/dummyless
  WCRAB: 100 runs, 100 matched to a cold reference
  WGRAPE: 100 runs, 100 matched to a cold reference

file                                                        lam  iter/state   si before    si after     improve
---------------------------------------------------------------------------------------------------------------
WCRAB_p40_det0.00_err0.00-lam0.0050-1.json                0.005        16.0    7.66e-02    1.29e-03       59.4x
WCRAB_p40_det0.00_err0.00-lam0.0050-2.json                0.005        15.0    7.66e-02    1.30e-03       59.1x
WCRAB_p40_det0.00_err0.00-lam0.0050-3.json                0.005        21.0    7.66e-02    1.28e-03       59.9x
WCRAB_p40_det0.00_err0.00-lam0.0050-4.json                0.005        15.0    7.66e-02    1.08e-03       71.0x
WCRAB_p40_det0.00_err0.00-lam0.0050-5.json                0.005        17.0    7.66e-02    1.04e-03       73.5x
WCRAB_p40_det0.00_err0.00-lam0.0100-1.json   